# Append Only

In [1]:
# ============================================================================
# CONFIGURATION
# ============================================================================
from urllib.parse import quote

storage_account_name = "cockroachcdc1768934658"  # ← Replace
storage_account_key = "***REMOVED***"      # ← Replace
storage_account_key_encoded = quote(storage_account_key, safe='')  # URL encode for use in connection strings
container_name = "changefeed-events"                 # ← Replace
target_catalog = "main"                              # ← Replace
target_schema = "robert_lee_crdb"                    # ← Replace


In [ ]:
# ============================================================================
# Azure Storage Authentication via Unity Catalog External Location
# ============================================================================
# ✅ External Location: cockroachcdc1768934658 is already configured
#
# IMPORTANT: External Location MUST use abfss:// (ADLS Gen2 with hierarchical namespace)
#   Correct:  abfss://changefeed-events@cockroachcdc1768934658.dfs.core.windows.net/
#   Wrong:    wasbs://changefeed-events@cockroachcdc1768934658.blob.core.windows.net/
#
# Verify permissions by running this in a SQL cell:
#   SHOW GRANTS ON EXTERNAL LOCATION `cockroachcdc1768934658`;
#
# You should see:
#   - READ FILES
#   - WRITE FILES (if needed)
#
# If not, request admin to run:
#   GRANT READ FILES ON EXTERNAL LOCATION `cockroachcdc1768934658` TO `your-email@domain.com`;
#
# Note: spark.conf.set() is NOT needed when using Unity Catalog External Locations

print("✅ Using Unity Catalog External Location for authentication")
print(f"   External Location: cockroachcdc1768934658")
print(f"   Expected URL: abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/")


✅ Using Unity Catalog External Location for authentication
   External Location: cockroachcdc1768934658
   Path: wasbs://changefeed-events@cockroachcdc1768934658.blob.core.windows.net/


In [ ]:
# ============================================================================
# Create append-only CDC events table using Spark Structured Streaming
# Note: For DLT pipeline version, see stream-changefeed-to-databricks-azure.md
# ============================================================================
from pyspark.sql import functions as F

# Use abfss:// (ADLS Gen2) for Unity Catalog External Location
# The storage account MUST have hierarchical namespace enabled
source_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/parquet/defaultdb/public/usertable/"
checkpoint_path = "/checkpoints/usertable/append_only"
target_table = f"{target_catalog}.{target_schema}.usertable_cdc_events"

# Read streaming data with Autoloader
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
    .option("recursiveFileLookup", "true")
    .load(source_path)
    .select(
        "*",
        F.when(F.col("__crdb__event_type") == "d", "DELETE")
         .otherwise("UPSERT")
         .alias("_cdc_operation"),
        F.col("__crdb__updated").alias("_cdc_timestamp")
    )
)

# Write all CDC events (no deduplication)
query = (df.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/data")
    .trigger(availableNow=True)  # Process all available data then stop
    .toTable(target_table)
)

print(f"✅ Streaming query started")
print(f"⏳ Processing data...")

query.awaitTermination()

print(f"\n✅ Completed! Query your data:")
print(f"   SELECT * FROM {target_table}")

SparkConnectGrpcException: (shaded.databricks.org.apache.hadoop.fs.azure.AzureException) shaded.databricks.org.apache.hadoop.fs.azure.AzureException: Unable to access container changefeed-events in account cockroachcdc1768934658.blob.core.windows.net using anonymous credentials, and no credentials found for them  in the configuration.

JVM stacktrace:
shaded.databricks.org.apache.hadoop.fs.azure.AzureException
	at shaded.databricks.org.apache.hadoop.fs.azure.AzureNativeFileSystemStore.createAzureStorageSession(AzureNativeFileSystemStore.java:1232)
	at shaded.databricks.org.apache.hadoop.fs.azure.AzureNativeFileSystemStore.initialize(AzureNativeFileSystemStore.java:562)
	at shaded.databricks.org.apache.hadoop.fs.azure.NativeAzureFileSystem.initialize(NativeAzureFileSystem.java:1397)
	at com.databricks.common.filesystem.LokiWASBFS.initialize(LokiWASBFS.scala:183)
	at com.databricks.sql.io.LokiFileSystem$.$anonfun$getLokiFS$1(LokiFileSystem.scala:122)
	at com.databricks.sql.io.FileSystemC